# AST 四数据集 M-Unified（seed 42）

这是直接M-Unified层级baseline，不接管Hanlin的single-dataset native baselines。四数据集共享同一个frozen AudioSet AST、biased Linear 768→256 projector与Level1/Crackle/Wheeze/Other heads，但prediction unit、split和eligibility保持原生边界。

## 冻结运行合同

seed=42；mono 16kHz；2秒window/1秒stride；短unit只zero-pad，长unit追加唯一end-aligned tail；homogeneous dataset batch=8；source-proportional；encoder冻结；FP32；无augmentation；Adam(lr=5e-5, weight_decay=1e-6)；50 reference epochs；每epoch validation；cosine per update，无warmup。

Level1使用2类CE；Crackle/Wheeze/Other使用masked BCE。每个node先在eligible rows内mean，再对当前active nodes等权mean。validation先得到每dataset eligible-node loss，再对active datasets等权mean选checkpoint，tie保留更早epoch。attribute thresholds仅由selected validation逐node max-F1确定，tie取更高threshold。outer test只在选模后推理一次。

## AST资产说明

用户优先指定的 `audioset_0.4593_runtime_cv4.pth` 当前本地文件实际为HTML而非可加载checkpoint。本运行使用同目录现有的AudioSet兼容AST权重 `hf_ast_legacy_compat.pth` 与现有source repo；不下载、不计算checksum。AST只使用2秒窗口对应的198-frame fbank grid，不再额外补到798帧。该定义命名为 `AST_2s_native_grid_v0`，是新的package，不能与旧798-frame padded P1结果直接称matched。未来Hanlin AST S-Native必须使用同一package，PH-Unified与M-Unified才可比较。运行设备优先MPS，当前环境不可用时使用CPU。

In [ ]:
from pathlib import Path
import json
from baseline.multidataset_pipeline.m_unified import run_ast_m_unified

ROOT = Path.cwd()
RESULT = ROOT / 'result/reproduce/unified/AST_M_Unified/seed_42'
summary_path = RESULT / 'run_summary.json'
summary = json.loads(summary_path.read_text()) if summary_path.is_file() else run_ast_m_unified(
    ROOT, RESULT, device_name='cpu', encoder_window_batch_size=8
)
summary

## 结果与claim boundary

主结果只报告各dataset的Level1/Crackle/Wheeze/Other metrics、confusion与support，以及dataset-macro和worst-dataset。`combined_pooled_auxiliary.json`仅作辅助诊断。HF是positive-only attribute supervision，不能解释为完整normal/abnormal detector；KAUH的Crep/Bronchial/I C B不进入训练或结论。结果必须保留 `AST_2s_native_grid_v0` package caveat，不与旧P1直接比较。